In [1]:
# Funciones basicas
import pandas as pd
import numpy as np

# Funciones de graficación
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Funciones de pronóstico y estadisticas
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.stattools import acf
import math
from math import sqrt
from scipy.stats import norm

# Funciones de manejo de fechas
from datetime import datetime, timedelta

# Funciones de manejo de texto
import re

# Funciones de manejo de archivos
import os
import io

# Funciones de manejo de excepciones
import sys
# Funciones para la interfaz de usuario
try:
    import streamlit as st
    USANDO_STREAMLIT = 'streamlit' in sys.modules
except ImportError:
    st = None
    USANDO_STREAMLIT = False

import warnings
warnings.filterwarnings(
    "ignore",
    message="pkg_resources is deprecated as an API",
    category=UserWarning,
    module="fs"
)
# Funciones de pronóstico avanzado
from statsforecast import StatsForecast
from statsforecast.models import HoltWinters

import matplotlib.pyplot as plt
import seaborn as sns

# Funciones de Apoyo de Carga de Datos

## Carga de datos de demanda

In [2]:
def cargar_demandas(ruta_demandas):

    # Lista para almacenar cada DataFrame
    dataframes = []

    # Itera sobre cada archivo en la carpeta
    for filename in os.listdir(ruta_demandas):
        if filename.endswith('2025.csv'):
            # Extrae "Producto" y "Regional" del nombre del archivo
            regional, año = filename.split('_')
            año = año.replace('.csv', '')
            
            # Carga el archivo y añade las columnas "Producto" y "Regional"
            df = pd.read_csv(os.path.join(ruta_demandas, filename))
            df['REGIONAL'] = regional
            print(f'Ultimo turno {regional}:',df['Turn'].max())
            # Agrega el DataFrame a la lista
            dataframes.append(df)

    # Concatena todos los DataFrames en uno solo
    df_agregado = pd.concat(dataframes, ignore_index=True)

    return df_agregado


In [3]:
def cargar_demandas_por_region(archivo_norte, archivo_centro, archivo_sur):
    
    """
    Carga y concatena los archivos de demanda por región (NORTE, CENTRO, SUR) 
    desde archivos subidos vía Streamlit.
    """

    dataframes = []

    archivos = {
        'NORTE': archivo_norte,
        'CENTRO': archivo_centro,
        'SUR': archivo_sur
    }

    for region, archivo in archivos.items():
        if archivo is not None:
            df = pd.read_csv(archivo)
            df['REGIONAL'] = region
            st.write(f"✅ Último turno cargado para {region}: {df['Turn'].max()}")
            dataframes.append(df)

    df_agregado = pd.concat(dataframes, ignore_index=True)

    return df_agregado

## Carga de datos maestros

In [4]:
def cargar_data_maestra(ruta_data_maestra):

    # Carga todas las hojas como diccionario
    hojas = pd.read_excel(ruta_data_maestra, sheet_name=None)  

    # Crear un DataFrame por cada hoja con nombre df_{nombre_hoja}
    for nombre_hoja, df in hojas.items():
        # Limpiar y estandarizar el nombre de la hoja
        nombre_limpio = re.sub(r'\W+', '_', nombre_hoja.lower())  # Minúsculas y reemplazo de no alfanuméricos por "_"
        globals()[f"df_{nombre_limpio}"] = df

    # (Opcional) Verificar los nombres creados
    print("Hojas cargadas:", [f"df_{re.sub(r'\\W+', '_', nombre.lower())}" for nombre in hojas.keys()])
    
    return df_bom_mp, df_m_d_o, df_transporte, df_almacenamiento

# Funciones de Apoyo para Preprocesamiento de Datos

### Producto terminado Parte 1

In [5]:
def preprocesar_datos_parte_1(df_agregado, productos):

    """
    Toma las demandas agregadas y la lista de productos,
    limpia los nombres de columnas, estandariza los nombres de las regionales,
    genera dos agregados adicionales:
    - 'CEDI': suma de CENTRO + SUR
    - 'MOTOTRAK': suma de NORTE + CENTRO + SUR
    Concatena estos agregados al DataFrame original y lo ordena.
    Devuelve el DataFrame final listo para análisis o modelado.
    """

    # Limpiar nombres de columnas
    df_agregado.columns = df_agregado.columns.str.replace(r"\s*\(Product\)", "", regex=True).str.strip()
    df_agregado['REGIONAL'] = df_agregado['REGIONAL'].str.upper()

    # Crear CEDI: CENTRO + SUR
    df_cedi = df_agregado[df_agregado['REGIONAL'].isin(['CENTRO', 'SUR'])].groupby('Turn')[productos].sum().reset_index()
    df_cedi['REGIONAL'] = 'CEDI'

    # Crear MOTOTRAK: NORTE + CENTRO + SUR
    df_mototrak = df_agregado[df_agregado['REGIONAL'].isin(['NORTE', 'CENTRO', 'SUR'])].groupby('Turn')[productos].sum().reset_index()
    df_mototrak['REGIONAL'] = 'MOTOTRAK'

    # Concatenar todo
    df_final = pd.concat([df_agregado, df_cedi, df_mototrak], ignore_index=True)

    # Mostar df
    return df_final

### Producto terminado Parte 2

In [6]:
def preprocesar_datos_parte_2(df_final):
    
    """    Transforma el DataFrame df_final para que las columnas de productos
    ('MOTO', 'CUATRIMOTO', 'TRACTOR') se conviertan en filas,
    manteniendo 'Turn' y 'REGIONAL' como columnas fijas.
    """

    # Transformar el DataFrame utilizando pd.melt
    df = pd.melt(
        df_final, 
        id_vars=['Turn', 'REGIONAL'],  # Columnas que permanecen fijas
        value_vars=['MOTO', 'CUATRIMOTO', 'TRACTOR'],  # Columnas que se convertirán en filas
        var_name='PRODUCTO',  # Nombre para la nueva columna de productos
        value_name='DEMANDA'  # Nombre para la nueva columna de valores
    )

    # Eliminar filas con DEMANDA nula (Tractor en Sur)
    df = df.dropna(subset='DEMANDA').reset_index(drop=True)

    # Visualizar el resultado
    return df

### Materia Prima Preprocesamiento BOM

In [7]:
def preprocesar_datos_mp(df_bom_mp):
    """
    Transforma el DataFrame df_bom_mp para que la columna de producto
    se conviertan en filas,
    manteniendo 'MATERIA_PRIMA' como columna fija.
    """
    # Seleccionar las columnas relevantes y renombrar 'PRODUCTO' a 'MATERIA_PRIMA'
    df_bom = df_bom_mp.rename(columns={'PRODUCTO':'MATERIA_PRIMA'}).iloc[:,:4]

    # Transformar el DataFrame utilizando pd.melt
    df_bom_vertical = df_bom.melt(id_vars=['MATERIA_PRIMA'], 
                                    var_name='PRODUCTO', 
                                    value_name='CANTIDAD')
    
    # Eliminar filas con CANTIDAD nula o cero
    df_bom_vertical = df_bom_vertical[df_bom_vertical['CANTIDAD'] != 0]
    
    return df_bom_vertical


### Materia Prima - Explosión de Materiales

In [8]:
def explosionar_mp(df, df_bom_vertical):

    """
    Explosiona el DataFrame df_mototrak con los datos de la BOM vertical
    para calcular el consumo de cada material por Turno.
    """
    
    # Filtrar df_mototrak
    df_mototrak = df[df['REGIONAL'] == 'MOTOTRAK'].copy()

    # Paso 1: Unir df_mototrak con df_bom_vertical por 'PRODUCTO'
    df_explosion = df_mototrak.merge(df_bom_vertical, on='PRODUCTO', how='left')

    # Paso 2: Calcular el consumo de cada material por Turn
    df_explosion['CONSUMO'] = df_explosion['DEMANDA'] * df_explosion['CANTIDAD']

    # Paso 3: Agrupar por Turno y Materia Prima
    df_consumo = (
        df_explosion
        .groupby(['Turn', 'MATERIA_PRIMA'], as_index=False)
        .agg({'CONSUMO': 'sum'})
        .rename(columns={'CONSUMO': 'DEMANDA_MATERIA_PRIMA'})
    )

    # Mostrar el DataFrame resultante
    return df_consumo

# Funciones de Ayuda para Gráficas de Demanda

## Función para graficar la demanda del producto terminado

In [9]:
def graficar_demanda_pt(df, colores_pt):

    """
    Crea un gráfico de líneas para la demanda de productos terminados
    por regionales y productos, utilizando Plotly.
    """

    # Listas de regionales y productos únicos
    regionales = df['REGIONAL'].unique().tolist()
    productos = df['PRODUCTO'].unique().tolist()

    # Crear figura 2x3
    fig = make_subplots(
        rows=2, cols=3, 
        subplot_titles=["NORTE", "CENTRO", "SUR", "CEDI (C+S)", "MOTOTRAK (N+C+S)", ""]
    )

    # Mapeo a subplot
    subplot_pos = {
        'NORTE': (1, 1),
        'CENTRO': (1, 2),
        'SUR': (1, 3),
        'CEDI': (2, 1),
        'MOTOTRAK': (2, 2)
    }

    # Mostrar leyenda solo en el primer subplot
    showlegend_flag = True

    # Trazar por cada regional
    for region in regionales:
        row, col = subplot_pos[region]
        df_region = df[df['REGIONAL'] == region]
        for producto in productos:
            df_sub = df_region[df_region['PRODUCTO'] == producto]
            if not df_sub.empty:
                fig.add_trace(
                    go.Scatter(
                        x=df_sub['Turn'], 
                        y=df_sub['DEMANDA'], 
                        mode='lines',
                        name=producto,
                        line=dict(color=colores_pt[producto]),
                        showlegend=showlegend_flag
                    ),
                    row=row, col=col
                )
        showlegend_flag = False  # Solo en el primer gráfico

    # Layout
    fig.update_layout(
        height=700, width=1200,
        title_text="Demanda por Regional y Agregados",
        showlegend=True,
        legend_title="Producto",
        template="ggplot2"
    )

    # Etiquetas comunes
    fig.update_xaxes(title_text="Turn", row=2, col=1)
    fig.update_xaxes(title_text="Turn", row=2, col=2)
    fig.update_yaxes(title_text="Demanda", row=1, col=1)
    fig.update_yaxes(title_text="Demanda", row=2, col=1)

    # Mostrar gráfico
    fig.show()

## Función para graficar los pronósticos de productos terminados

In [10]:
def graficar_pronosticos_pt(df, resultados_pt, colores_pt):

    # Listas de regionales y productos únicos
    regionales = df['REGIONAL'].unique().tolist()
    productos = df['PRODUCTO'].unique().tolist()

    # Crear figura 2x3
    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=["NORTE", "CENTRO", "SUR", "CEDI (C+S)", "MOTOTRAK (N+C+S)", ""]
    )

    # Posiciones de subplots
    subplot_pos = {
        'NORTE': (1, 1),
        'CENTRO': (1, 2),
        'SUR': (1, 3),
        'CEDI': (2, 1),
        'MOTOTRAK': (2, 2)
    }

    # Mostrar leyenda solo una vez
    showlegend_flag = True

    # Agregar trazos de demanda real y pronóstico
    for region in regionales:
        row, col = subplot_pos[region]
        df_region = df[df['REGIONAL'] == region]

        for producto in productos:
            # 1. Demanda real
            df_sub = df_region[df_region['PRODUCTO'] == producto]
            clave = (region, producto)

            if not df_sub.empty:
                # Si hay pronóstico, ajustar la longitud del histórico
                if clave in resultados_pt:
                    pronostico_final = resultados_pt[clave]["pronostico_final"]
                    mejor_modelo = resultados_pt[clave]["mejor_modelo"]

                    lags = len(pronostico_final)  # Cantidad de pasos de pronóstico
                    df_sub = df_sub.tail(52 + lags)  # Cortar a los últimos 52 + lags

                    fig.add_trace(
                        go.Scatter(
                            x=df_sub['Turn'],
                            y=df_sub['DEMANDA'],
                            mode='lines',
                            name=producto,
                            line=dict(color=colores_pt[producto]),
                            showlegend=showlegend_flag
                        ),
                        row=row, col=col
                    )

                    if not pronostico_final.empty:
                        fig.add_trace(
                            go.Scatter(
                                x=pronostico_final.index,
                                y=pronostico_final[mejor_modelo],
                                mode='lines',
                                name=f"{producto} ({mejor_modelo})",
                                line=dict(dash='dot', color=colores_pt[producto]),
                                showlegend=showlegend_flag
                            ),
                            row=row, col=col
                        )
                else:
                    # Si no hay pronóstico, igual limitar a últimos 52 datos
                    df_sub = df_sub.tail(52)
                    fig.add_trace(
                        go.Scatter(
                            x=df_sub['Turn'],
                            y=df_sub['DEMANDA'],
                            mode='lines',
                            name=producto,
                            line=dict(color=colores_pt[producto]),
                            showlegend=showlegend_flag
                        ),
                        row=row, col=col
                    )

        showlegend_flag = False  # Solo mostrar en el primer subplot

    # Layout final
    fig.update_layout(
        height=700, width=1200,
        title_text="Demanda Real y Pronóstico por Regional y Producto",
        showlegend=True,
        legend_title="Producto / Modelo",
        template="ggplot2"
    )

    # Etiquetas ejes
    fig.update_xaxes(title_text="Turn", row=2, col=1)
    fig.update_xaxes(title_text="Turn", row=2, col=2)
    fig.update_yaxes(title_text="Demanda", row=1, col=1)
    fig.update_yaxes(title_text="Demanda", row=2, col=1)

    return fig

## Función para graficar los pronosticos de materia prima

In [11]:
def generar_colores_mp(elementos):
    """
    Asigna colores únicos a cada elemento (producto o materia prima).
    Usa una paleta de colores de Plotly.
    
    Parámetro:
    - elementos: lista o conjunto de nombres
    
    Retorna:
    - diccionario {elemento: color}
    """
    elementos = sorted(list(set(elementos)))
    paleta = px.colors.qualitative.Set2  # Puedes cambiar por Set1, Set2, Plotly, etc.
    n_colores = len(paleta)

    colores_mp = {
        elemento: paleta[i % n_colores]
        for i, elemento in enumerate(elementos)
    }

    return colores_mp

In [12]:
def graficar_pronosticos_mp(df, resultados_mp, colores_mp, lags):
    """
    Grafica series de consumo real y pronóstico para materias primas.

    Parámetros:
    - df: DataFrame con columnas ['Turn', 'MATERIA_PRIMA', 'DEMANDA_MATERIA_PRIMA']
    - resultados_por_serie: dict con claves = materia prima y valores con 'pronostico_final' y 'mejor_modelo'
    - colores_mp: dict {materia_prima: color}
    - lags: número de pasos de pronóstico
    """

    elementos = sorted(df['MATERIA_PRIMA'].unique())
    n = len(elementos)
    cols = 3
    rows = math.ceil(n / cols)

    # Dividir nombres largos con salto de línea si exceden cierto número de caracteres
    def ajustar_titulo(texto, max_len=30):
        return "<br>".join(texto[i:i+max_len] for i in range(0, len(texto), max_len))

    titulos = elementos

    fig = make_subplots(
        rows=rows, cols=cols,
        subplot_titles=titulos
    )

    showlegend_flag = True

    for i, materia in enumerate(elementos):
        row = (i // cols) + 1
        col = (i % cols) + 1

        fila = df[df['MATERIA_PRIMA'] == materia].tail(52 + lags)
        color = colores_mp.get(materia, None)

        fig.add_trace(
            go.Scatter(
                x=fila['Turn'],
                y=fila['DEMANDA_MATERIA_PRIMA'],
                mode='lines',
                name=materia,
                line=dict(color=color),
                showlegend=showlegend_flag
            ),
            row=row, col=col
        )

        if materia in resultados_mp:
            pronostico_final = resultados_mp[materia]["pronostico_final"]
            mejor_modelo = resultados_mp[materia]["mejor_modelo"]
            lags = len(pronostico_final)  # Cantidad de pasos de pronóstico
            fig.add_trace(
                go.Scatter(
                    x=pronostico_final.index,
                    y=pronostico_final[mejor_modelo],
                    mode='lines',
                    name=f"{materia} ({mejor_modelo})",
                    line=dict(dash='dot', color=color),
                    showlegend=showlegend_flag
                ),
                row=row, col=col
            )

        showlegend_flag = False

    # Disminuir tamaño de fuente de títulos individuales
    for anotacion in fig['layout']['annotations']:
        anotacion['font'] = dict(size=11)

    fig.update_layout(
        height=300 * rows, width=1200,
        title_text="Consumo y Pronóstico por Materia Prima",
        showlegend=False,
        legend_title="Materia Prima / Modelo",
        template="ggplot2",
        font=dict(size=12)  # Solo afecta ejes, leyenda, título general
    )
    fig.update_xaxes(title_text="Turn")
    fig.update_yaxes(title_text="Demanda")

    return fig

# Funciones de Ayuda para Selección de pronósticos de Producto terminado

## Creación de diccionario con series de tiempo producto-regional

In [13]:
# Crear un diccionario con cada serie de tiempo de demanda por producto y regional
def crear_dicc_pt(df):

    """
    Crea un diccionario donde las claves son tuplas (REGIONAL, PRODUCTO)
    y los valores son Series de DEMANDA indexadas por Turn.
    """
    
    series_dict_pt = {
        (reg, prod): serie
        for reg in df['REGIONAL'].unique()
        for prod in df['PRODUCTO'].unique()
        if not (serie := df[(df['REGIONAL'] == reg) & (df['PRODUCTO'] == prod)]
                    .set_index('Turn')['DEMANDA']
                    .sort_index()).empty
    }

    return series_dict_pt

## Creación de diccionario con series de tiempo materia prima en mototrak

In [14]:
# Crear un diccionario con cada serie de tiempo de demanda por materia prima
def crear_dicc_mp(df_consumo):

    """
    Crea un diccionario con df de materia prima explosionada,
    y los valores son Series de DEMANDA indexadas por Turn.
    """

    series_dict_mp = {
        materia: serie
        for materia in df_consumo['MATERIA_PRIMA'].unique()
        if not (serie := df_consumo[df_consumo['MATERIA_PRIMA'] == materia]
                        .set_index('Turn')['DEMANDA_MATERIA_PRIMA']
                        .sort_index()).empty
    }

    return series_dict_mp

## Backtesting
Se hará backtesting desde n periodos hacia atras y generando múltiples pronósticos hacia adelante (lags)

In [15]:
def crear_pronosticos_generico(series_dict, periodos_atras=48, lags=6):
    """
    Aplica modelos de pronóstico sobre un diccionario de series univariadas.
    Funciona tanto para productos terminados como materias primas.
    """

    turnos = next(iter(series_dict.values())).index.tolist()
    rango_turnos = turnos[-(periodos_atras + 1):]
    resultados_por_serie = {}

    # Widgets dinámicos solo si estás en Streamlit
    progreso = st.empty() if USANDO_STREAMLIT else None
    barra = st.progress(0) if USANDO_STREAMLIT else None
    total = len(series_dict)

    for i, (clave, serie) in enumerate(series_dict.items()):
        if USANDO_STREAMLIT:
            progreso.markdown(f"👨‍💻 Analizando `{clave}`...")
            barra.progress((i + 1) / total)
        else:
            print(f"👨‍💻 Analizando {clave}")

        resultados_hw, resultados_hw_13 = [], []
        resultados_pm_3, resultados_pm_6, resultados_pm_12 = [], [], []

        for j, fecha_corte in enumerate(rango_turnos):
            serie_corte = serie[serie.index <= fecha_corte].copy()
            indice_real = serie_corte.index.copy()
            serie_corte.index = pd.RangeIndex(start=0, stop=len(serie_corte))

            inicio_pronostico = fecha_corte + 1
            fin_pronostico = inicio_pronostico + lags - 1

            if len(serie_corte) >= 10:
                try:
                    modelo_hw = ExponentialSmoothing(serie_corte, trend='add', seasonal=None).fit()
                    forecast_hw = modelo_hw.forecast(lags)
                    forecast_hw.index = range(inicio_pronostico, fin_pronostico + 1)

                    modelo_hw_13 = ExponentialSmoothing(
                        serie_corte, trend='add', seasonal='add', seasonal_periods=13
                    ).fit()
                    forecast_hw_13 = modelo_hw_13.forecast(lags)
                    forecast_hw_13.index = range(inicio_pronostico, fin_pronostico + 1)
                except:
                    forecast_hw = pd.Series([np.nan] * lags, index=range(inicio_pronostico, fin_pronostico + 1))
                    forecast_hw_13 = pd.Series([np.nan] * lags, index=range(inicio_pronostico, fin_pronostico + 1))
            else:
                forecast_hw = pd.Series([np.nan] * lags, index=range(inicio_pronostico, fin_pronostico + 1))
                forecast_hw_13 = pd.Series([np.nan] * lags, index=range(inicio_pronostico, fin_pronostico + 1))

            serie_corte.index = indice_real

            pm_3 = serie_corte.rolling(3).mean().iloc[-1] if len(serie_corte) >= 3 else np.nan
            pm_6 = serie_corte.rolling(6).mean().iloc[-1] if len(serie_corte) >= 6 else np.nan
            pm_12 = serie_corte.rolling(12).mean().iloc[-1] if len(serie_corte) >= 12 else np.nan

            pm_3_series = pd.Series([pm_3] * lags, index=range(inicio_pronostico, fin_pronostico + 1))
            pm_6_series = pd.Series([pm_6] * lags, index=range(inicio_pronostico, fin_pronostico + 1))
            pm_12_series = pd.Series([pm_12] * lags, index=range(inicio_pronostico, fin_pronostico + 1))

            demanda_real = serie.loc[inicio_pronostico:fin_pronostico]

            df_comb = pd.DataFrame({
                'real': demanda_real,
                'hw': forecast_hw,
                'hw_13': forecast_hw_13,
                'pm_3': pm_3_series,
                'pm_6': pm_6_series,
                'pm_12': pm_12_series,
            })

            if j < len(rango_turnos) - 1:
                df_comb = df_comb.dropna(subset=['real'])
                resultados_hw.append(df_comb[['real', 'hw']])
                resultados_hw_13.append(df_comb[['real', 'hw_13']])
                resultados_pm_3.append(df_comb[['real', 'pm_3']])
                resultados_pm_6.append(df_comb[['real', 'pm_6']])
                resultados_pm_12.append(df_comb[['real', 'pm_12']])
            else:
                pronostico_final_hw = df_comb[['real', 'hw']]
                pronostico_final_hw_13 = df_comb[['real', 'hw_13']]
                pronostico_final_pm_3 = df_comb[['real', 'pm_3']]
                pronostico_final_pm_6 = df_comb[['real', 'pm_6']]
                pronostico_final_pm_12 = df_comb[['real', 'pm_12']]

        modelos = {
            'hw': (resultados_hw, pronostico_final_hw),
            'hw_13': (resultados_hw_13, pronostico_final_hw_13),
            'pm_3': (resultados_pm_3, pronostico_final_pm_3),
            'pm_6': (resultados_pm_6, pronostico_final_pm_6),
            'pm_12': (resultados_pm_12, pronostico_final_pm_12),
        }

        metricas_modelos = {}
        for nombre_modelo, (resultados, _) in modelos.items():
            if resultados:
                df_resultado = pd.concat(resultados)
                df_resultado["error"] = df_resultado["real"] - df_resultado[nombre_modelo]
                df_resultado["error_abs"] = df_resultado["error"].abs()
                suma_real = df_resultado["real"].sum()
                mae_porc = df_resultado["error_abs"].sum() / suma_real
                sesgo_porc = df_resultado["error"].sum() / suma_real
                score_porc = mae_porc + abs(sesgo_porc)
                rmse = np.sqrt((df_resultado["error"] ** 2).mean())
            else:
                mae_porc = np.nan
                sesgo_porc = np.nan
                score_porc = np.inf
                rmse = np.nan

            metricas_modelos[nombre_modelo] = {
                "mae_porc": mae_porc,
                "sesgo_porc": sesgo_porc,
                "score_porc": round(score_porc, 3),
                "rmse": rmse
            }

        df_metricas = pd.DataFrame(metricas_modelos).T.sort_values("score_porc")
        mejor_modelo = df_metricas.index[0]
        pronostico_final = modelos[mejor_modelo][1]

        resultados_por_serie[clave] = {
            "mejor_modelo": mejor_modelo,
            "metricas": df_metricas,
            "pronostico_final": pronostico_final
        }

    return resultados_por_serie

## Funciones de Ayuda para la generacion de reportes

### Reporte Producto Terminado

In [16]:
def generar_resumen_pt(resultados_pt):
    """
    Genera un DataFrame resumen con los mejores modelos y pronósticos finales   
    """

    resumen_filas = []

    for (regional, producto), datos in resultados_pt.items():
        mejor_modelo = datos['mejor_modelo']
        metricas = datos['metricas']
        pronostico_final = datos['pronostico_final']

        rmse_val = metricas.loc[mejor_modelo, 'rmse']
        score_val = metricas.loc[mejor_modelo, 'score_porc']
        pronostico = pronostico_final[mejor_modelo]

        fila = {
            'REGIONAL': regional,
            'PRODUCTO': producto,
            'MODELO': mejor_modelo.upper(),  
            'SCORE_PORC': f"{round(score_val * 100, 1)}%",                     
            'RMSE': round(rmse_val, 1),
        }

        for turno, valor in pronostico.items():
            fila[turno] = round(valor, 0) if pd.notna(valor) else np.nan

        resumen_filas.append(fila)

    df_resumen = pd.DataFrame(resumen_filas)

    cols_fijas = ['REGIONAL', 'PRODUCTO','MODELO', 'SCORE_PORC', 'RMSE']
    cols_turnos = sorted([col for col in df_resumen.columns if isinstance(col, (int, str)) and col not in cols_fijas])
    df_resumen = df_resumen[cols_fijas + cols_turnos]

    return df_resumen

### Reporte Materia Prima

In [17]:
def generar_resumen_mp(resultados_mp):
    """
    Genera un DataFrame resumen con los mejores modelos y pronósticos finales para materias primas
    """

    resumen_filas = []

    for producto, datos in resultados_mp.items():
        mejor_modelo = datos['mejor_modelo']
        metricas = datos['metricas']
        pronostico_final = datos['pronostico_final']

        rmse_val = metricas.loc[mejor_modelo, 'rmse']
        score_val = metricas.loc[mejor_modelo, 'score_porc']
        pronostico = pronostico_final[mejor_modelo]

        fila = {
            'PRODUCTO': producto,
            'MODELO': mejor_modelo.upper(),
            'SCORE_PORC': f"{round(score_val * 100, 1)}%",
            'RMSE': round(rmse_val, 1),
        }

        for turno, valor in pronostico.items():
            fila[turno] = round(valor, 0) if pd.notna(valor) else np.nan

        resumen_filas.append(fila)

    df_resumen = pd.DataFrame(resumen_filas)

    cols_fijas = ['PRODUCTO', 'MODELO', 'SCORE_PORC', 'RMSE']
    cols_turnos = sorted([col for col in df_resumen.columns if col not in cols_fijas])
    df_resumen = df_resumen[cols_fijas + cols_turnos]

    return df_resumen

# Script de Ejecución Parte 1 - Producto Terminado

In [18]:
# Define la carpeta donde están los archivos
ruta_demandas = 'dataset/'
df_agregado = cargar_demandas(ruta_demandas)

# Define los productos a considerar
productos = ['MOTO', 'CUATRIMOTO', 'TRACTOR']

# Preprocesar los datos parte 1
df_final = preprocesar_datos_parte_1(df_agregado, productos)

# Preprocesar los datos parte 2
df = preprocesar_datos_parte_2(df_final)

# Mostrar el DataFrame final
df

# Crear diccionario con series de tiempo por producto y regional
#series_dict_pt = crear_dicc_pt(df)

Ultimo turno centro: 0
Ultimo turno norte: 0
Ultimo turno sur: 0


,Turn,REGIONAL,PRODUCTO,DEMANDA
0,-207,CENTRO,MOTO,14.0
1,-206,CENTRO,MOTO,13.0
2,-205,CENTRO,MOTO,10.0
3,-204,CENTRO,MOTO,7.0
4,-203,CENTRO,MOTO,20.0
...,...,...,...,...
2907,-4,MOTOTRAK,TRACTOR,17.0
2908,-3,MOTOTRAK,TRACTOR,8.0
2909,-2,MOTOTRAK,TRACTOR,13.0
2910,-1,MOTOTRAK,TRACTOR,13.0


In [19]:
# df has columns ['Turn','REGIONAL','PRODUCTO','DEMANDA']
df_long = df.copy()
df_long['unique_id'] = df_long['REGIONAL'] + '_' + df_long['PRODUCTO']
df_long = df_long.rename(columns={'Turn': 'ds', 'DEMANDA': 'y'})
df_long = df_long[['unique_id','ds','y']].sort_values(['unique_id','ds'])
df_long

,unique_id,ds,y
1664,CEDI_CUATRIMOTO,-207,5.0
1665,CEDI_CUATRIMOTO,-206,16.0
1666,CEDI_CUATRIMOTO,-205,18.0
1667,CEDI_CUATRIMOTO,-204,14.0
1668,CEDI_CUATRIMOTO,-203,23.0
...,...,...,...
619,SUR_MOTO,-4,16.0
620,SUR_MOTO,-3,19.0
621,SUR_MOTO,-2,15.0
622,SUR_MOTO,-1,21.0


In [28]:
# set up Holt-Winters model (season_length=52 for weekly seasonality)
models = [HoltWinters(season_length=13, error_type="A", alias="hw_13"),
          HoltWinters(season_length=26, error_type="A", alias="hw_26")
          ]


sf = StatsForecast(
    models=models,
    freq=1,      # spacing between your integer 'ds' values
    n_jobs=-1
)

# fit and cross‑validate
sf.fit(df_long)



StatsForecast(models=[hw_13,hw_26])

In [29]:
result=sf.fitted_[0,0].model_
print(result.keys())
print(result['fit'])

dict_keys(['loglik', 'aic', 'bic', 'aicc', 'mse', 'amse', 'fit', 'residuals', 'components', 'm', 'nstate', 'fitted', 'states', 'par', 'sigma2', 'n_params', 'method', 'actual_residuals'])
results(x=array([ 3.83063891e-04,  1.14956927e-04,  3.14829290e-04,  1.75582701e+01,
        1.76519727e-02, -1.41198182e+00, -8.38023153e-01, -8.72130003e-01,
       -5.96446632e-01, -8.76169600e-01, -4.74829806e-01,  6.82593374e-01,
        9.56372243e-01,  2.56306402e-02,  1.41184263e+00, -5.01729091e-01,
        1.63164778e+00]), fn=1628.065754031435, nit=1001, simplex=array([[ 3.16608365e-04,  1.10509896e-04,  3.30456364e-04,
         1.75642695e+01,  1.75485726e-02, -1.40411862e+00,
        -8.68313202e-01, -8.83465955e-01, -5.83935009e-01,
        -9.12391464e-01, -4.42558826e-01,  6.81347349e-01,
         9.50867442e-01,  2.56440470e-02,  1.45045061e+00,
        -5.01072264e-01,  1.61548013e+00],
       [ 3.51943491e-04,  1.07268149e-04,  3.76319773e-04,
         1.75646275e+01,  1.83583158e-02

In [30]:
Y_hat = sf.forecast(df=df_long, h=6, fitted=True)
Y_hat

,unique_id,ds,hw_13,hw_26
0,CEDI_CUATRIMOTO,1,22.021197,21.451998
1,CEDI_CUATRIMOTO,2,22.806249,23.411316
2,CEDI_CUATRIMOTO,3,20.690272,20.370253
3,CEDI_CUATRIMOTO,4,22.620696,22.833854
4,CEDI_CUATRIMOTO,5,21.255020,22.374496
...,...,...,...,...
79,SUR_MOTO,2,19.988814,19.522763
80,SUR_MOTO,3,17.402371,16.689376
81,SUR_MOTO,4,17.212396,16.988992
82,SUR_MOTO,5,16.146631,16.550975


In [31]:
values=sf.forecast_fitted_values()
values.head()

,unique_id,ds,y,hw_13,hw_26
0,CEDI_CUATRIMOTO,-207,5.0,18.439145,17.965991
1,CEDI_CUATRIMOTO,-206,16.0,19.218529,19.864330
2,CEDI_CUATRIMOTO,-205,18.0,17.099656,16.801581
3,CEDI_CUATRIMOTO,-204,14.0,19.029413,19.270786
4,CEDI_CUATRIMOTO,-203,23.0,17.656537,18.780293


In [44]:
# Tomar el primer unique_id
primer_id = values["unique_id"].unique()[11]

# Filtrar el DataFrame
df_filtrado = values[values["unique_id"] == primer_id]

# Reestructurar datos para plotly (long format)
values_plot = df_filtrado.melt(
    id_vars=["ds"], 
    value_vars=["y", "hw_13", "hw_26"],
    var_name="Serie", 
    value_name="Valor"
)

# Graficar con estilo ggplot2
fig = px.line(
    values_plot, 
    x="ds", 
    y="Valor", 
    color="Serie",       # cada serie tendrá color distinto
    markers=True,
    title=f"Serie real vs Holt-Winters (HW_13 y HW_26) - {primer_id}",
    template="ggplot2"   # estilo similar a ggplot2
)

fig.show()


In [45]:
Y_hat.head(60)

,unique_id,ds,hw_13,hw_26
0,CEDI_CUATRIMOTO,1,22.021197,21.451998
1,CEDI_CUATRIMOTO,2,22.806249,23.411316
2,CEDI_CUATRIMOTO,3,20.690272,20.370253
3,CEDI_CUATRIMOTO,4,22.620696,22.833854
4,CEDI_CUATRIMOTO,5,21.255020,22.374496
5,CEDI_CUATRIMOTO,6,22.200019,21.911144
6,CEDI_MOTO,1,43.978802,43.746701
7,CEDI_MOTO,2,45.530699,44.264476
8,CEDI_MOTO,3,38.815111,36.609717
9,CEDI_MOTO,4,36.820172,36.260014


In [46]:
cv_df = sf.cross_validation(
    df=df_long,
    h=6,           # 6-week forecast horizon
    n_windows=26,  # number of rolling folds
    step_size=1,   # move the cut-off by 1 week each time (overlapping windows)
    refit=True
)

In [56]:
cv_df.tail(60)

,unique_id,ds,cutoff,y,hw_13,hw_26
2124,SUR_MOTO,-14,-15,23.0,20.535750,20.499796
2125,SUR_MOTO,-13,-15,19.0,20.717240,20.443782
2126,SUR_MOTO,-12,-15,20.0,19.974463,20.112262
2127,SUR_MOTO,-11,-15,19.0,19.693325,20.320748
2128,SUR_MOTO,-10,-15,15.0,17.374680,18.658839
2129,SUR_MOTO,-9,-15,17.0,16.974864,17.096220
2130,SUR_MOTO,-13,-14,19.0,20.828536,20.374144
2131,SUR_MOTO,-12,-14,20.0,19.999742,20.087777
2132,SUR_MOTO,-11,-14,19.0,19.684079,20.191961
2133,SUR_MOTO,-10,-14,15.0,17.470857,18.719500


In [51]:
def evaluar_cv_por_id(cv_df: pd.DataFrame, modelos: list[str]) -> dict:
    """
    Calcula MAE%, sesgo% y score% por cada series (unique_id) y por cada modelo,
    tanto de forma global como agrupado por lag (horizonte).

    Parámetros
    ----------
    cv_df : DataFrame
        DataFrame devuelto por StatsForecast.cross_validation con columnas:
        ['unique_id','ds','cutoff','y', model_1, model_2, ...].
    modelos : list of str
        Nombres de las columnas que contienen las predicciones de cada modelo.

    Devuelve
    --------
    resultados : dict
        Diccionario indexado por unique_id.  Cada entrada a su vez es un diccionario
        donde las claves son los nombres de modelo y los valores son una tupla
        (mae_porc, sesgo_porc, score_porc, df_lags).  El DataFrame `df_lags` tiene
        las columnas ['Lags','mae_porc','sesgo_porc','score_porc'].
    """
    resultados = {}
    # Recorrer cada serie individual
    for uid, df_uid in cv_df.groupby('unique_id'):
        # Calcular lag numérico y etiqueta Lag_n
        df_uid = df_uid.copy()
        df_uid['lag_number'] = df_uid['ds'] - df_uid['cutoff']
        df_uid['Lags'] = 'Lag_' + df_uid['lag_number'].astype(int).astype(str)

        resultados[uid] = {}
        for mod in modelos:
            # Calcular errores
            df_uid[f'error_{mod}']     = df_uid['y'] - df_uid[mod]
            df_uid[f'error_abs_{mod}'] = df_uid[f'error_{mod}'].abs()

            # Métricas globales
            suma_y = df_uid['y'].sum()
            mae_porc   = df_uid[f'error_abs_{mod}'].sum() / suma_y
            sesgo_porc = df_uid[f'error_{mod}'].sum()     / suma_y
            score_porc = mae_porc + abs(sesgo_porc)

            # Métricas por lag
            agg = df_uid.groupby('Lags').apply(
                lambda g: pd.Series({
                    'mae_porc':   g[f'error_abs_{mod}'].sum() / g['y'].sum(),
                    'sesgo_porc': g[f'error_{mod}'].sum()     / g['y'].sum()
                })
            ).reset_index()

            agg['score_porc'] = agg['mae_porc'] + agg['sesgo_porc'].abs()

            # Guardar resultados de este modelo para este unique_id
            resultados[uid][mod] = (mae_porc, sesgo_porc, score_porc, agg)
    return resultados

In [54]:
# supondremos que tus columnas de modelo se llaman 'hw_13' y 'hw_26'
resumen = evaluar_cv_por_id(cv_df, ['hw_13','hw_26'])

# Muestra resultados
for uid, modelos_res in resumen.items():
    print(f'Serie {uid}')
    for mod, (mae, sesgo, score, df_lags) in modelos_res.items():
        print(f'  Modelo: {mod}')
        print(f'    MAE%:   {mae:.4f}')
        print(f'    Sesgo%: {sesgo:.4f}')
        print(f'    Score%: {score:.4f}')
        print('    Métricas por lag:')
        print(df_lags)
        print()

Serie CEDI_CUATRIMOTO
  Modelo: hw_13
    MAE%:   0.1414
    Sesgo%: 0.0020
    Score%: 0.1433
    Métricas por lag:
    Lags  mae_porc  sesgo_porc  score_porc
0  Lag_1  0.134790    0.007697    0.142487
1  Lag_2  0.133652    0.000442    0.134094
2  Lag_3  0.141904    0.006602    0.148506
3  Lag_4  0.140812   -0.002450    0.143263
4  Lag_5  0.144734    0.010323    0.155057
5  Lag_6  0.152422   -0.011024    0.163446

  Modelo: hw_26
    MAE%:   0.1473
    Sesgo%: -0.0027
    Score%: 0.1500
    Métricas por lag:
    Lags  mae_porc  sesgo_porc  score_porc
0  Lag_1  0.136546    0.002494    0.139040
1  Lag_2  0.140356   -0.003569    0.143925
2  Lag_3  0.149622    0.002442    0.152064
3  Lag_4  0.146912   -0.004355    0.151267
4  Lag_5  0.155531    0.001620    0.157151
5  Lag_6  0.154756   -0.015080    0.169836

Serie CEDI_MOTO
  Modelo: hw_13
    MAE%:   0.1079
    Sesgo%: -0.0138
    Score%: 0.1217
    Métricas por lag:
    Lags  mae_porc  sesgo_porc  score_porc
0  Lag_1  0.107550   -0.0267

C:\Users\MSI\AppData\Local\Temp\ipykernel_27800\3423279392.py:43: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

C:\Users\MSI\AppData\Local\Temp\ipykernel_27800\3423279392.py:43: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

C:\Users\MSI\AppData\Local\Temp\ipykernel_27800\3423279392.py:43: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of p

In [253]:
# Define la carpeta donde están los archivos
ruta_demandas = 'dataset/'
df_agregado = cargar_demandas(ruta_demandas)

# Define los productos a considerar
productos = ['MOTO', 'CUATRIMOTO', 'TRACTOR']

# Preprocesar los datos parte 1
df_final = preprocesar_datos_parte_1(df_agregado, productos)

# Preprocesar los datos parte 2
df = preprocesar_datos_parte_2(df_final)

# Mostrar el DataFrame final
df

# Definir cololres para los productos terminados
colores_pt = {
    'MOTO': 'salmon',
    'CUATRIMOTO': 'navy',
    'TRACTOR': 'darkcyan'
}
# Graficar la demanda de producto terminado
#graficar_demanda_pt(df, colores_pt)

# Crear diccionario con series de tiempo por producto y regional
series_dict_pt = crear_dicc_pt(df)

# Realizar pronósticos para las series de tiempo de producto terminado
resultados_pt = crear_pronosticos_generico(series_dict_pt, periodos_atras=48, lags=6)

# Graficar los pronósticos de producto terminado
fig = graficar_pronosticos_pt(df, resultados_pt, colores_pt)
fig.show()
# Generar resumen de los resultados de pronósticos de producto terminado
df_resumen = generar_resumen_pt(resultados_pt)
display(df_resumen)

2025-07-31 15:40:54.975 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:40:54.975 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:40:54.976 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:40:54.976 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:40:54.976 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:40:54.977 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:40:54.977 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:40:54.977 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

Ultimo turno centro: 0
Ultimo turno norte: 0
Ultimo turno sur: 0


2025-07-31 15:40:58.997 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:40:58.998 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:40:58.998 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:40:58.998 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:40:58.999 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:40:58.999 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:41:03.453 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:41:03.453 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

,REGIONAL,PRODUCTO,MODELO,SCORE_PORC,RMSE,1,2,3,4,5,6
0,CENTRO,MOTO,HW,16.6%,4.0,22.0,22.0,22.0,22.0,22.0,22.0
1,CENTRO,CUATRIMOTO,PM_6,14.4%,1.9,12.0,12.0,12.0,12.0,12.0,12.0
2,CENTRO,TRACTOR,PM_12,27.6%,3.5,11.0,11.0,11.0,11.0,11.0,11.0
3,NORTE,MOTO,HW_13,11.4%,1.7,18.0,18.0,16.0,16.0,14.0,14.0
4,NORTE,CUATRIMOTO,PM_12,26.1%,2.7,9.0,9.0,9.0,9.0,9.0,9.0
5,NORTE,TRACTOR,HW,47.4%,1.8,3.0,3.0,3.0,3.0,3.0,3.0
6,SUR,MOTO,HW_13,8.5%,1.6,20.0,20.0,18.0,17.0,16.0,16.0
7,SUR,CUATRIMOTO,PM_12,34.5%,3.2,7.0,7.0,7.0,7.0,7.0,7.0
8,CEDI,MOTO,HW_13,10.5%,4.2,44.0,45.0,39.0,37.0,38.0,33.0
9,CEDI,CUATRIMOTO,PM_12,15.1%,3.7,19.0,19.0,19.0,19.0,19.0,19.0


# Script de Ejecución Parte 2 - Materia Prima

In [254]:
# Cargar todas las hojas del archivo Excel
ruta_data_maestra = r'dataset\INFO_MAESTRA_BOM_TIEMPOS.xlsx'

# Cargar los DataFrames de la data maestra
df_bom_mp, df_m_d_o, df_transporte, df_almacenamiento = cargar_data_maestra(ruta_data_maestra)

# Preprocesar los datos de materia prima
df_bom_vertical = preprocesar_datos_mp(df_bom_mp)

# Explosionar los datos de materia prima
df_consumo = explosionar_mp(df, df_bom_vertical)

# Crear un diccionario con series de tiempo de materia prima
series_dict_mp = crear_dicc_mp(df_consumo)

# Generar pronósticos para las series de tiempo de materia prima
resultados_mp = crear_pronosticos_generico(series_dict_mp, periodos_atras=48, lags=12)

# Generar colores para las materias primas
colores_mp = generar_colores_mp(df_bom_vertical['MATERIA_PRIMA'].unique())

# Graficar los pronósticos de materia prima
fig = graficar_pronosticos_mp(df_consumo, resultados_mp, colores_mp, lags=12)
fig.show()
# Generar resumen de los resultados de pronósticos de materia prima
generar_resumen_mp(resultados_mp)



2025-07-31 15:44:11.595 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:44:11.597 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:44:11.597 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:44:11.599 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:44:11.601 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:44:11.602 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:44:11.602 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:44:11.604 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

Hojas cargadas: ['df_bom_mp', 'df_m_d_o', 'df_transporte', 'df_almacenamiento']


2025-07-31 15:44:49.311 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:44:49.315 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:44:49.319 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:44:49.321 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:44:49.323 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:44:49.325 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:45:14.454 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:45:14.455 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

,PRODUCTO,MODELO,SCORE_PORC,RMSE,1,2,3,4,5,6,7,8,9,10,11,12
0,BLOQUE DE EJES NEGRO,PM_12,8.1%,31.8,329.0,329.0,329.0,329.0,329.0,329.0,329.0,329.0,329.0,329.0,329.0,329.0
1,BLOQUE REDONDO HEMBRA-HEMBRA AMARILLO,HW_13,8.5%,16.6,211.0,218.0,193.0,185.0,181.0,174.0,178.0,184.0,190.0,203.0,213.0,223.0
2,BLOQUE REDONDO MACHO-MACHO AMARILLO,HW_13,8.5%,40.3,518.0,535.0,468.0,448.0,440.0,412.0,424.0,447.0,455.0,489.0,520.0,549.0
3,BLOQUE REDONDO SENCILLO AMARILLO,HW_13,8.5%,36.3,470.0,486.0,427.0,409.0,401.0,379.0,390.0,408.0,418.0,447.0,473.0,498.0
4,COLUMNA NEGRA,HW_13,10.7%,9.7,89.0,91.0,83.0,80.0,77.0,78.0,79.0,79.0,85.0,88.0,90.0,93.0
5,CUARTO LADRILLO AZUL,PM_12,8.0%,13.3,138.0,138.0,138.0,138.0,138.0,138.0,138.0,138.0,138.0,138.0,138.0,138.0
6,DINTEL DOBLE AMARILLO,PM_12,8.1%,25.3,262.0,262.0,262.0,262.0,262.0,262.0,262.0,262.0,262.0,262.0,262.0,262.0
7,DINTEL SIMPLE AMARILLO,PM_12,8.1%,17.8,181.0,181.0,181.0,181.0,181.0,181.0,181.0,181.0,181.0,181.0,181.0,181.0
8,DINTEL TRIPLE AMARILLO,HW,22.0%,11.7,43.0,43.0,43.0,43.0,43.0,43.0,43.0,43.0,43.0,43.0,43.0,43.0
9,EJE HEXAGONAL 10 CM,PM_12,9.4%,10.2,85.0,85.0,85.0,85.0,85.0,85.0,85.0,85.0,85.0,85.0,85.0,85.0


# Front End Streamlit

In [255]:

st.set_page_config(page_title="App de Pronósticos Mototrak", layout="wide")

st.title("App de Pronósticos para Producto Terminado y Materia Prima")
#pestaña_pt, pestaña_mp = st.tabs(["Pronósticos PT", "Pronósticos MP"])
seccion = st.sidebar.radio("Selecciona sección", ["Pronósticos PT", "Pronósticos MP"])
# ----------------------------
# PESTAÑA PRODUCTO TERMINADO
# ----------------------------
if seccion == "Pronósticos PT":
    st.subheader("Cargar archivos de demanda por región")
    archivo_norte = st.file_uploader("Archivo demanda NORTE", type=["csv"])
    archivo_centro = st.file_uploader("Archivo demanda CENTRO", type=["csv"])
    archivo_sur = st.file_uploader("Archivo demanda SUR", type=["csv"])

    periodos_atras_pt = st.number_input("Periodos hacia atrás para backtesting (PT)", min_value=1, max_value=60, value=12)
    lags_pt = st.number_input("Cantidad de periodos a pronosticar (lags PT)", min_value=1, max_value=24, value=6)

    if archivo_norte and archivo_centro and archivo_sur:
        productos = ['MOTO', 'CUATRIMOTO', 'TRACTOR']
        df_agregado = cargar_demandas_por_region(archivo_norte, archivo_centro, archivo_sur)
        df_final = preprocesar_datos_parte_1(df_agregado, productos)
        df = preprocesar_datos_parte_2(df_final)
        st.session_state["df"] = df

        colores_pt = {'MOTO': 'salmon', 'CUATRIMOTO': 'navy', 'TRACTOR': 'darkcyan'}
        series_dict_pt = crear_dicc_pt(df)

        if st.button("Generar pronóstico de PT"):
          
            resultados_pt = crear_pronosticos_generico(series_dict_pt, periodos_atras_pt, lags_pt)
            df_resumen_pt = generar_resumen_pt(resultados_pt)

            st.session_state['resultados_pt'] = resultados_pt
            st.session_state['df_resumen_pt'] = df_resumen_pt

            fig = graficar_pronosticos_pt(df, resultados_pt, colores_pt)
            st.session_state['fig_pt'] = fig

    # Mostrar resultados si ya existen
    if 'df_resumen_pt' in st.session_state:
        st.subheader("Resumen del pronóstico PT")
        st.dataframe(st.session_state['df_resumen_pt'], use_container_width=True)

        # Reconstruir gráfica si no está en session_state
        if 'fig_pt' not in st.session_state:
            st.session_state['fig_pt'] = graficar_pronosticos_pt(
                st.session_state['df'],
                st.session_state['resultados_pt'],
                {'MOTO': 'salmon', 'CUATRIMOTO': 'navy', 'TRACTOR': 'darkcyan'}
            )

        st.plotly_chart(st.session_state['fig_pt'], use_container_width=True)

        buffer_pt = io.BytesIO()
        st.session_state['df_resumen_pt'].to_excel(buffer_pt, index=False)
        st.download_button(
            "📥 Descargar resumen PT en Excel",
            data=buffer_pt.getvalue(),
            file_name="resumen_pt.xlsx"
        )

# ----------------------------
# PESTAÑA MATERIA PRIMA
# ----------------------------
elif seccion == "Pronósticos MP":
    st.subheader("Cargar archivo maestro de datos")
    archivo_maestro = st.file_uploader("Archivo Excel (Info Maestra)", type=["xlsx"])

    if archivo_maestro:
        df_bom_mp, df_m_d_o, df_transporte, df_almacenamiento = cargar_data_maestra(archivo_maestro)
        df_bom_vertical = preprocesar_datos_mp(df_bom_mp)
        st.session_state["df_bom_vertical"] = df_bom_vertical  # 💾 Guardar en session_state

        if st.button("Ejecutar explosión de materiales"):
            try:
                if "df" not in st.session_state:
                    st.warning("Primero debes generar el pronóstico de Producto Terminado.")
                    st.stop()

                df = st.session_state["df"]
                df_consumo = explosionar_mp(df, df_bom_vertical)
                st.session_state["df_consumo"] = df_consumo  # 💾 Guardar en session_state
                st.success("Explosión realizada con éxito")
            except Exception as e:
                st.error(f"Error durante la explosión de materiales: {e}")

        # Parámetros visibles siempre que haya datos disponibles
        if "df_consumo" in st.session_state and "df_bom_vertical" in st.session_state:
            periodos_atras_mp = st.number_input("Periodos hacia atrás para backtesting (MP)", min_value=1, max_value=60, value=12)
            lags_mp = st.number_input("Cantidad de periodos a pronosticar (lags MP)", min_value=1, max_value=24, value=6)

            if st.button("Generar pronóstico de MP"):
                try:
                    df_consumo = st.session_state["df_consumo"]
                    df_bom_vertical = st.session_state["df_bom_vertical"]

                    series_dict_mp = crear_dicc_mp(df_consumo)                  
                    resultados_mp = crear_pronosticos_generico(series_dict_mp, periodos_atras_mp, lags_mp)
                    df_resumen_mp = generar_resumen_mp(resultados_mp)

                    st.session_state['resultados_mp'] = resultados_mp
                    st.session_state['df_resumen_mp'] = df_resumen_mp

                    colores_mp = generar_colores_mp(df_bom_vertical['MATERIA_PRIMA'].unique())
                    st.session_state['colores_mp'] = colores_mp
                    fig = graficar_pronosticos_mp(df_consumo, resultados_mp, colores_mp, lags=lags_mp)
                    
                    st.session_state['fig_mp'] = fig
                    #st.dataframe(df_resumen_mp, use_container_width=True)



                except Exception as e:
                    st.error(f"Error durante el pronóstico: {e}")

    # Mostrar resultados si ya existen
    if 'df_resumen_mp' in st.session_state:
        st.subheader("Resumen del pronóstico MP")
        st.dataframe(st.session_state['df_resumen_mp'], use_container_width=True)

        # Reconstruir gráfica si no está en session_state
        if 'fig_mp' not in st.session_state:
            st.session_state['fig_mp'] = graficar_pronosticos_mp(
                st.session_state['df'],
                st.session_state['resultados_mp'],
                st.session_state['colores_mp']
            )

        st.plotly_chart(st.session_state['fig_mp'], use_container_width=True)

        buffer_mp = io.BytesIO()
        st.session_state['df_resumen_mp'].to_excel(buffer_mp, index=False)
        st.download_button(
            "📥 Descargar resumen MP en Excel",
            data=buffer_mp.getvalue(),
            file_name="resumen_mp.xlsx"
        )


2025-07-31 15:56:46.463 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:56:46.464 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:56:46.465 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:56:46.466 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:56:46.467 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:56:46.468 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:56:46.468 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-31 15:56:46.469 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

# Analisis de Capacidad en Mototrak

## Explosion de carga en minutos

In [256]:
def preprocesar_tiempos(df_m_d_o):
    # Seleccionar columnas
    df_tiempos = df_m_d_o.iloc[:,:4]

    # Convertir a Vertical
    df_tiempos_vertical = df_tiempos.melt(id_vars=['PROCESO'], 
                                    var_name='PRODUCTO', 
                                    value_name='MINUTOS')
    return df_tiempos_vertical

In [257]:
def extraer_pronosticos_finales(resultados_pt, productos):

    # Crear lista para guardar los DataFrames
    pronosticos_mototrak = []

    for producto in productos:
        datos = resultados_pt[('MOTOTRAK', producto)]
        modelo = datos['mejor_modelo']
        df_pronostico = datos['pronostico_final'][[modelo]].copy()
        df_pronostico.columns = ['CANTIDAD']
        df_pronostico['PRODUCTO'] = producto
        df_pronostico['TURNO'] = df_pronostico.index
        pronosticos_mototrak.append(df_pronostico)

    # Unir todos los pronósticos
    df_mototrak = pd.concat(pronosticos_mototrak, ignore_index=True)

    # Redondear cantidad al entero más cercano
    df_mototrak['CANTIDAD'] = df_mototrak['CANTIDAD'].round().astype(int)

    # Reordenar columnas
    df_mototrak = df_mototrak[['TURNO', 'PRODUCTO', 'CANTIDAD']]

    return df_mototrak


In [258]:
def explosion_minutos(df_tiempos_vertical, df_mototrak, df_m_d_o):
    # Unir pronósticos con tiempos por producto y proceso
    df_explosion = df_mototrak.merge(df_tiempos_vertical, on='PRODUCTO', how='left')

    # Calcular minutos requeridos por turno y proceso
    df_explosion['MINUTOS_TOTALES'] = df_explosion['CANTIDAD'] * df_explosion['MINUTOS']

    # Agrupar por TURNO y PROCESO
    df_grafico = df_explosion.groupby(['TURNO', 'PROCESO'], as_index=False)['MINUTOS_TOTALES'].sum()

    # Unir capacidad por proceso
    df_grafico = df_grafico.merge(df_m_d_o[['PROCESO', 'CAPACIDAD_INTERNA_MINUTOS']], on='PROCESO', how='left')

    return df_grafico

In [259]:
def graficar_explosion_minutos(df_grafico):
    # Crear lista de procesos únicos
    procesos = df_grafico['PROCESO'].unique()

    # Crear figura con 4 subplots (2 filas, 2 columnas)
    fig = make_subplots(rows=2, cols=2, subplot_titles=procesos)

    # Mapeo de posición de subplots
    posiciones = {(0): (1,1), (1): (1,2), (2): (2,1), (3): (2,2)}

    # Agregar cada gráfico por proceso
    for i, proceso in enumerate(procesos):
        row, col = posiciones[i]
        df_proc = df_grafico[df_grafico['PROCESO'] == proceso]

        # Línea de minutos requeridos
        fig.add_trace(
            go.Scatter(
                x=df_proc['TURNO'],
                y=df_proc['MINUTOS_TOTALES'],
                mode='lines+markers',
                name='Demanda',
                showlegend=False
            ),
            row=row, col=col
        )

        # Línea de capacidad (constante)
        fig.add_trace(
            go.Scatter(
                x=df_proc['TURNO'],
                y=[df_proc['CAPACIDAD_INTERNA_MINUTOS'].iloc[0]] * len(df_proc),
                mode='lines',
                name='Capacidad',
                line=dict(dash='dash'),
                showlegend=False
            ),
            row=row, col=col
        )

    # Ajustes de diseño
    fig.update_layout(
        height=700,
        width=1000,
        title_text="Minutos requeridos vs capacidad por proceso",
        template='ggplot2'
    )

    #fig.show()

    return fig

In [260]:
graficar_explosion_minutos(df_grafico)

# EOQ en Mototrak

## Definición de variables

In [333]:
# Data frame con dimensiones
dimensiones = pd.DataFrame({'PRODUCTO':['MOTO', 'CUATRIMOTO', 'TRACTOR'],'WIDTH':[0.8,1.1,1.2],'DEPTH':[2.2,1.2,2.5],'HEIGHT':[1.4,1.2,1.3]})
dimensiones['DIMENSION'] = dimensiones['WIDTH']*dimensiones['DEPTH']*dimensiones['HEIGHT']	

# Valores unitarios (Precios) de los productos
valores_unitarios = {'MOTO': 7_000_000, 'CUATRIMOTO': 9_000_000, 'TRACTOR': 11_000_000}

# Semana por año
semanas_por_ano = 52

## Determinacion de Costos de los Productos

### Costos de materia prima

In [334]:
# Fusionar bom con el costo de caa materia prima
df_costos_mp = df_bom_vertical.merge(df_bom_mp.rename(columns={'PRODUCTO':'MATERIA_PRIMA'})[['MATERIA_PRIMA', 'COSTO']], how='left', on='MATERIA_PRIMA')

# Multiplicar cantidad por costo
df_costos_mp['COSTO_MP'] = df_costos_mp['CANTIDAD'] * df_costos_mp['COSTO']

# Agrupar por producto y sumar
df_costos_mp_sku = df_costos_mp.groupby('PRODUCTO')['COSTO_MP'].sum().reset_index()

# Mostrar df
df_costos_mp_sku

,PRODUCTO,COSTO_MP
0,CUATRIMOTO,3788000
1,MOTO,3927000
2,TRACTOR,4140000


### Costos de mano de obra

In [335]:
# Fusionar tiempos con costo
df_costos_mdo = df_tiempos_vertical.merge(df_m_d_o[['PROCESO','COSTO_VBLE_INTERNO']], how='left', on='PROCESO')

# Multiplicar minutos por costo
df_costos_mdo['COSTO_MDO'] = df_costos_mdo['MINUTOS'] *df_costos_mdo['COSTO_VBLE_INTERNO']

# Agrupar por producto y sumar
df_costos_mdo_sku = df_costos_mdo.groupby('PRODUCTO')['COSTO_MDO'].sum().reset_index()

# Mostrar df
df_costos_mdo_sku

,PRODUCTO,COSTO_MDO
0,CUATRIMOTO,1378500
1,MOTO,1642500
2,TRACTOR,1665000


### Calular costos totales

In [336]:
# Fusionar costos de materia prima y costos de mano de obra
df_costos_sku = df_costos_mp_sku.merge(df_costos_mdo_sku, how='left', on='PRODUCTO')

# Sumar los dos componentes
df_costos_sku['COSTO_TOTAL'] = df_costos_sku['COSTO_MP'] + df_costos_sku['COSTO_MDO']

# Adicionar precio del producto
df_costos_sku['PRECIO'] = df_costos_sku['PRODUCTO'].map(valores_unitarios)

# Calcular utilidad unitaria
df_costos_sku['UTILIDAD'] = df_costos_sku['PRECIO'] - df_costos_sku['COSTO_TOTAL']

# Mostrar df
df_costos_sku

,PRODUCTO,COSTO_MP,COSTO_MDO,COSTO_TOTAL,PRECIO,UTILIDAD
0,CUATRIMOTO,3788000,1378500,5166500,9000000,3833500
1,MOTO,3927000,1642500,5569500,7000000,1430500
2,TRACTOR,4140000,1665000,5805000,11000000,5195000


## Costos de Ordenar (***A***)

El costo de ordenar puede obtenerse como la suma de los costos fijos de cada proceso en Mototrak

In [337]:
A = df_m_d_o['COSTO_FIJO_INTERNO'].sum()
print('Costo de ordenar A:', A)

Costo de ordenar A: 2700000


## Costos de Almacenar (***r***)

Para calcular el costo de almacenar se hara una estimación de cuanto cuesta almacenar un m3 por semana, luego este valor se multiplica por la dimensión de cada producto y esto por 52 semanas al año. 
Como segundo elmento tenemos el costo de oportunidad que será calculado con base en una tasa interna de retorno de 12%.

In [338]:
# Tasa interna de referencia
tasa_retorno_int = 0.12

### Costo de almacenar una unidad por una semana

In [339]:
# Filtrar y extraer el valor de m3 por semana
costo_m3_sem = df_almacenamiento.loc[df_almacenamiento['ESLABON'] == 'Mototrak', 'COSTO_VBLE_$xM3'].values[0]

# Filtrar y extraer costo fijo de arrendamiento
costo_fijo_sem = df_almacenamiento.loc[df_almacenamiento['ESLABON'] == 'Mototrak', 'COSTO_FIJO_SEM'].values[0]

# Filtrar y extraer capacidad en m3 de mototrak
capacidad_m3 = df_almacenamiento.loc[df_almacenamiento['ESLABON'] == 'Mototrak', 'CAPACIDAD_M3'].values[0]

# Calcluar costo total variable por m3 por semana
costo_total_m3_sem = costo_fijo_sem / capacidad_m3 + costo_m3_sem


# Mostrar valores
print('Costo_m3_sem:',costo_m3_sem)
print('Costo_fijo_sem:',costo_fijo_sem)
print('Capacidad_m3:',capacidad_m3)
print('Costo total m3 por sem:',costo_total_m3_sem)

Costo_m3_sem: 2500
Costo_fijo_sem: 1000000
Capacidad_m3: 2880
Costo total m3 por sem: 2847.222222222222


### Costo variable de almacenar una unidad de cada sku por año

In [340]:
# Adicionar costo total a df dimensiones
dimensiones['COSTO_VBLE_ANUAL'] = dimensiones['DIMENSION'] * costo_total_m3_sem * semanas_por_ano

# Mostrar df
dimensiones

,PRODUCTO,WIDTH,DEPTH,HEIGHT,DIMENSION,COSTO_VBLE_ANUAL
0,MOTO,0.8,2.2,1.4,2.464,364808.888889
1,CUATRIMOTO,1.1,1.2,1.2,1.584,234520.000000
2,TRACTOR,1.2,2.5,1.3,3.900,577416.666667


In [341]:
# Fusionar costos variables al año con df_costos_sku
df_costos_sku = df_costos_sku[['PRODUCTO', 'COSTO_TOTAL', 'UTILIDAD']].merge(dimensiones[['PRODUCTO', 'COSTO_VBLE_ANUAL']], how= 'left', on='PRODUCTO').copy()

# Motrar df
df_costos_sku

,PRODUCTO,COSTO_TOTAL,UTILIDAD,COSTO_VBLE_ANUAL
0,CUATRIMOTO,5166500,3833500,234520.000000
1,MOTO,5569500,1430500,364808.888889
2,TRACTOR,5805000,5195000,577416.666667


### Costo de oportunidad

In [342]:
# Calculo de costo de oportunidad como multiplicación de tasa interna de retorno por costo del producto 
df_costos_sku['COSTO_OPORTUNIDAD_ANUAL'] = df_costos_sku['COSTO_TOTAL'] * tasa_retorno_int

### Calculo de ***r*** como porcentaje del costo del producto al año

In [343]:
# Calcular costo total de almacenar una unidad al año
df_costos_sku['r_ANUAL'] = (df_costos_sku['COSTO_VBLE_ANUAL'] + df_costos_sku['COSTO_OPORTUNIDAD_ANUAL']) / df_costos_sku['COSTO_TOTAL']

# Mostrar df
df_costos_sku

,PRODUCTO,COSTO_TOTAL,UTILIDAD,COSTO_VBLE_ANUAL,COSTO_OPORTUNIDAD_ANUAL,r_ANUAL
0,CUATRIMOTO,5166500,3833500,234520.000000,619980.0,0.165392
1,MOTO,5569500,1430500,364808.888889,668340.0,0.185501
2,TRACTOR,5805000,5195000,577416.666667,696600.0,0.219469


### Calculo de demanda anual

La demanda anual se saca del historico explosionado a nivel de mototrak teniendo en cuenta los ultimos 52 turnos

In [344]:
# Fitrar df por regional Mototrak y limitando a ultimos 52 turnos
df_demanda_eoq = df[(df['REGIONAL']=='MOTOTRAK') & (df['Turn'] > -52)].groupby('PRODUCTO')['DEMANDA'].sum().reset_index()

## Calculo del EOQ

In [345]:
# Fusionar df_costos_sku con demanda 
df_costos_sku = df_costos_sku.merge(df_demanda_eoq, how= 'left', on='PRODUCTO')

# Aplicacion de la formula
df_costos_sku['EOQ'] = np.sqrt((2*df_costos_sku['DEMANDA']*A)/(df_costos_sku['COSTO_TOTAL']*df_costos_sku['r_ANUAL']))

# Calcular la cobertura semanal estimada del EOQ
df_costos_sku['COB_EOQ'] = df_costos_sku['EOQ'] / (df_costos_sku['DEMANDA']/52)

# Mostrar df
df_costos_sku 

,PRODUCTO,COSTO_TOTAL,UTILIDAD,COSTO_VBLE_ANUAL,COSTO_OPORTUNIDAD_ANUAL,r_ANUAL,DEMANDA,EOQ,COB_EOQ
0,CUATRIMOTO,5166500,3833500,234520.000000,619980.0,0.165392,1523.0,98.104922,3.349610
1,MOTO,5569500,1430500,364808.888889,668340.0,0.185501,2747.0,119.824259,2.268242
2,TRACTOR,5805000,5195000,577416.666667,696600.0,0.219469,739.0,55.966938,3.938134


### Requerimiento en minutos del EOQ

In [347]:
# Filtrar solo EOQ
minutos_EOQ = df_costos_sku[['PRODUCTO','EOQ']]

# Unir EOQ con tabla de tiempos
minutos_EOQ = minutos_EOQ.merge(df_tiempos_vertical, on='PRODUCTO', how='left').copy()

# Calcular minutos requeridos por lote EOQ
minutos_EOQ['MINUTOS_TOTALES'] = minutos_EOQ['EOQ'] * minutos_EOQ['MINUTOS']

# Resumen de minutos por proceso
df_minutos_proceso = (
    minutos_EOQ.groupby('PROCESO', as_index=False)['MINUTOS_TOTALES']
          .sum()
          .rename(columns={'MINUTOS_TOTALES': 'MINUTOS_REQUERIDOS'})
)

# Mostrar minutos requeridos para fabricar EOQ
df_minutos_proceso

,PROCESO,MINUTOS_REQUERIDOS
0,ARMAR BASE,5477.922385
1,ARMAR CHASIS,44011.604490
2,ASIENTO - VOLANTE,26303.645075
3,RUEDAS - ORUGA,66797.853886


# Modelo de Inventarios PT para Norte, Centro y Sur

## Calculo de R comun por regional

### Costo de Ordenar ***A***

Se estima como el costo del flete comun para cada regional

In [350]:
flete_norte = df_transporte.loc[df_transporte['DESTINO'] == 'Mototrak Norte', 'COSTO_FIJO'].values[0]
flete_centro = df_transporte.loc[df_transporte['DESTINO'] == 'Mototrak Centro', 'COSTO_FIJO'].values[0]
flete_sur = df_transporte.loc[df_transporte['DESTINO'] == 'Mototrak Sur', 'COSTO_FIJO'].values[0]

print('Costo ordenar común Norte:', flete_norte)
print('Costo ordenar común Centro:', flete_centro)
print('Costo ordenar común Sur:', flete_sur)

Costo ordenar común Norte: 2500000
Costo ordenar común Centro: 1000000
Costo ordenar común Sur: 1000000


In [348]:
df_transporte

,ORIGEN,DESTINO,TIPO_TRANSPORTE,CACPACIDAD_M3,N°_VEHICULOS_DISPONIBLES,COSTO_FIJO,LEAD_TIME_TRANSPORTE_SEM
0,Proveedor Azul,Mototrak,Truck (Camión),72,1,1000000,1
1,Proveedor Amarillo,Mototrak,Truck (Camión),144,1,2000000,1
2,Proveedor Gris - Negro,Mototrak,Ship (Barco),1944,1,8000000,4
3,Proveedor Gris - Negro,Mototrak,Plane (Aéreo),6,1,1200000,1
4,Mototrak,Mototrak Norte,Truck (Camión),72,2,2500000,1
5,Mototrak,CEDI,Truck (Camión),144,2,1000000,1
6,CEDI,Mototrak Centro,Truck (Camión),72,2,1000000,1
7,CEDI,Mototrak Sur,Truck (Camión),72,2,1000000,1


### Calculo del costo de almacenar (***r***)